## __Tentative de fine tuning de mobilenetV2__

In [2]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(".."))

from transformers import AutoImageProcessor, AutoModelForImageClassification
from torchmetrics.classification import MulticlassF1Score
from lib.data.preprocessing import TorchPreprocessor
from lib.data.dataset import BeeDataset
from lib.data.train_val_split import train_val_split
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch.optim as optim
from lib.data.data_augmentation import data_augmented_loader

import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"L'entraînement se fera sur : {device}")

L'entraînement se fera sur : cuda


In [6]:
# Load model directly from Hugging Face
num_labels = 50
model_name = "google/efficientnet-b0"

base_processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name, 
                                                        num_labels=num_labels,
                                                        ignore_mismatched_sizes=True,
                                                        use_safetensors=True)

Some weights of EfficientNetForImageClassification were not initialized from the model checkpoint at google/efficientnet-b0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([50]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 1280]) in the checkpoint and torch.Size([50, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Le preprocessor fourni ne pad pas, ce qui nous fait perdre des abeilles. On se propose donc de remplacer cette partie du code

In [7]:
mobile_net_processor_parameters= {
    "mean" : base_processor.image_mean,
    "std" : base_processor.image_std,
    "crop_size" : (base_processor.size["height"], base_processor.size["width"])
}

light_train_preprocessor = TorchPreprocessor(
    normalize=True,
    resize_method="pad",
    augmentation="light",
    target_size=mobile_net_processor_parameters["crop_size"],
    mean=mobile_net_processor_parameters["mean"],
    std=mobile_net_processor_parameters["std"],
    interpolation_method="BICUBIC"
)

heavy_train_preprocessor = TorchPreprocessor(
    normalize=True,
    resize_method="pad",
    augmentation="heavy",
    target_size=mobile_net_processor_parameters["crop_size"],
    mean=mobile_net_processor_parameters["mean"],
    std=mobile_net_processor_parameters["std"],
    interpolation_method="BICUBIC"
)   

val_preprocessor = TorchPreprocessor(
    normalize=True,
    resize_method="pad",
    augmentation="none",
    target_size=mobile_net_processor_parameters["crop_size"],
    mean=mobile_net_processor_parameters["mean"],
    std=mobile_net_processor_parameters["std"],
    interpolation_method="BICUBIC"
)

train_loader, val_loader = data_augmented_loader(
    mean=mobile_net_processor_parameters["mean"],
    std=mobile_net_processor_parameters["std"],
    target_size=mobile_net_processor_parameters["crop_size"],
    train_preprocessor_light=light_train_preprocessor,
    train_preprocessor_heavy=heavy_train_preprocessor,
    val_preprocessor=val_preprocessor
)

Train prêt : 6417 images (avec augmentation ciblée)
Val prête  : 1582 images (sans augmentation)


In [8]:
def evaluate(model, dataloader, device, metric_fn):
    model.eval()
    metric_fn.reset()
    total_loss = 0
    criterion = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            logits = outputs.logits
            
            # Calcul de la perte
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            # Calcul du F1
            preds = torch.argmax(logits, dim=-1)
            metric_fn.update(preds, labels)
            
    avg_loss = total_loss / len(dataloader)
    f1_score = metric_fn.compute()
    return avg_loss, f1_score

In [9]:
# 1. Configuration (Backbone gelé, Tête active)
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

# Filtrer les paramètres pour l'optimiseur (plus propre pour la mémoire)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.01)
criterion = torch.nn.CrossEntropyLoss()

# 2. Boucle d'entraînement
model.train()
num_epochs = 20

train_f1_metric = MulticlassF1Score(num_classes=num_labels, average='macro').to(device)
test_f1_metric = MulticlassF1Score(num_classes=num_labels, average='macro').to(device)
model.to(device)

for epoch in range(num_epochs):
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)
    
    epoch_loss = 0
    
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        logits = outputs.logits
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs.logits, dim=-1)
        train_f1_metric.update(preds, labels)
        
        epoch_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.3f}", f1=f"{train_f1_metric.compute():.3f}")
        
    test_loss, test_f1 = evaluate(model, val_loader, device, test_f1_metric)

    # Affichage final de l'époque
    train_f1 = train_f1_metric.compute()
    print(f"-> Fin Epoch {epoch+1}")
    print(f"   TRAIN | Loss: {epoch_loss/len(train_loader):.4f} | F1: {train_f1:.4f}")
    print(f"   TEST  | Loss: {test_loss:.4f} | F1: {test_f1:.4f}")
    
    # Reset la métrique d'entraînement pour la prochaine époque
    train_f1_metric.reset()
    test_f1_metric.reset()

    # Score final de l'epoch
    print(f"-> Fin Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f} | F1: {train_f1:.4f}")

Epoch [1/20]:   0%|          | 0/201 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
test_dataset = BeeDataset(train=False, transform=val_preprocessor)

model.eval()

all_results = [] 
with torch.no_grad():
    for images, ids in test_dataset:
        images = images.to(device)
        outputs = model(images.unsqueeze(0))
        
        logits = outputs.logits
        
        predicted_idx = torch.argmax(logits, dim=-1).item()
        
        final_label = predicted_idx
        
        all_results.append({"id": ids, "label": final_label})

submission_results = pd.DataFrame(all_results)

# Sauvegarde finale
submission_results.to_csv("submission1.csv", index=False)
print(f"✅ Terminé ! {len(submission_results)} prédictions sauvegardées.")

KeyboardInterrupt: 

In [ ]:
class JITWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x):
        # On extrait uniquement le Tensor des logits pour satisfaire le JIT
        outputs = self.model(x)
        return outputs.logits

model.eval()
model.to("cpu") # L'export est plus stable sur CPU
wrapper = JITWrapper(model)

dummy_input = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(wrapper, dummy_input)

traced_model.save("models/mobilenet_v2_bees_jit.pt")

In [ ]:
loaded_jit_model = torch.jit.load("models/mobilenet_v2_bees_jit.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_jit_model.to(device)
loaded_jit_model.eval()

with torch.no_grad():
    # Assure-toi que les images sont aussi sur le même device
    images = images.to(device)
    results = loaded_jit_model(images)